# ECB Exchange Rates ETL Pipeline: Initial Database Setup

### Imports and ECB dataset codes

This section defines the Python libraries required by the ETL process and the dataset codes used to query the European Central Bank (ECB) Data API.

#### Imported libraries

- `requests`: sends HTTP requests to the ECB API.
- `xml.etree.ElementTree`: parses XML responses returned by the API.
- `StringIO`: creates in-memory text buffers that can be used as temporary CSV-like files.
- `psycopg`: connects Python to PostgreSQL.

#### ECB dataset constants

The script stores ECB dataset paths as constants so they can be reused across the extraction process. Each constant identifies a specific time series collection available through the ECB Data API.

Defined datasets:

- `EXTERNAL_NET_DEBT`: external net debt in absolute terms.
- `EXTERNAL_NET_DEBT_PERCENTAGE`: external net debt as a percentage of GDP.
- `GROSS_EXTERNAL_DEBT`: gross external debt in absolute terms.
- `INFLATION`: HICP inflation rate for headline inflation and selected components.
- `EXCHANGE_RATE`: daily exchange rates of selected currencies against the euro.

Using constants improves readability, reduces duplication, and makes the code easier to maintain.

In [ ]:
import requests
import xml.etree.ElementTree as ET
from io import StringIO
import psycopg

In [ ]:
# External net debt in absolute terms
EXTERNAL_NET_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR._T._X.N.ALL"
# External net debt as a percentage of GDP
EXTERNAL_NET_DEBT_PERCENTAGE = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.NE.FA._T.FNED._Z.EUR_R_B1GQ._T._X.N.ALL"
# Gross external debt in absolute terms
GROSS_EXTERNAL_DEBT = "BPS/Q.N.AT+BE+HR+CY+EE+FI+FR+DE+GR+IE+IT+LV+LT+LU+MT+NL+PT+SK+SI+ES+I9.W1.S1.S1.LE.L.FA._T.FGED._Z.EUR._T._X.N.ALL"
# HICP inflation rate for headline and selected components
INFLATION = "ICP/M.DE+FR+ES+IT+NL+BE+AT+PT+FI+IE+GR+SK+SI+EE+LV+LT+LU+CY+MT+U2.N.000000+FOOD00+NRGY00+SERV00+GOODS00+CP0000+ALCBEV+TOBACC+CLTHES+HOUSNG+FURNSH+HEALTH+TRANS+COMMUN+RECREA+EDUCAT+RESTAU+MISCEL.4.ANR"
# Daily exchange rates of selected currencies against the euro
EXCHANGE_RATE = "EXR/D.BGN+CZK+DKK+HUF+PLN+RON+SEK+CHF+NOK+GBP+ISK+USD+JPY+AUD+CAD+NZD.EUR.SP00.A"

## Main ETL function

The `new_database()` function orchestrates the full ETL workflow for exchange rate data.

### Workflow steps

1. Open a connection to the PostgreSQL database.
2. Create the `exchange_rates` fact table if it does not already exist.
3. Request exchange rate data from the ECB API.
4. Transform the XML response into an in-memory buffer.
5. Load the transformed data into the database.
6. Create and populate the `dim_currency` dimension table.

### Output

The function returns a confirmation message once the process finishes successfully.

### Note

This implementation closes the database connection and cursor manually. A more robust version could use context managers (`with`) to ensure resources are always released, even if an error occurs during execution.

In [ ]:
def new_database():

    # Open a connection to the PostgreSQL database
    conn = psycopg.connect(
    host=,
    port=,
    dbname=,
    user=,
    password=
    )

    # Create a cursor to execute database queries
    cursor = conn.cursor()

    #Create the exchange rates fact table in the database
    create_fact_table_exchange_rates(conn, cursor)

    # Request new exchange rate data from the external API
    data_exchange_rate_bce = api_extract_data(EXCHANGE_RATE)

    # Transform the API response into an in-memory CSV buffer
    buffer_exchange_rate_data = create_csv_exchange_rate(data_exchange_rate_bce)

    # Load the processed exchange rate data into the database
    load_exchange_rates(buffer_exchange_rate_data, conn, cursor)

    # Create and populate the currency dimension table
    dimension_table_currency(conn)

    # Close database resources
    cursor.close()
    conn.close()

    return "Process completed successfully"

#### Creating the exchange rates fact table

The `fact_table_exchange_rates()` function creates the `exchange_rates` table in PostgreSQL if it does not already exist.

##### Table structure

The table stores daily exchange rate observations and includes the following fields:

- `id`: surrogate primary key generated automatically.
- `date`: observation date.
- `currency`: three-letter currency code.
- `rate`: exchange rate value.

##### Constraint design

A unique constraint is defined on the combination of `date` and `currency`. This ensures that each currency has at most one exchange rate observation per date.

In [ ]:
def create_fact_table_exchange_rates(conn, cursor):
    
    # SQL statement to create the fact table for exchange rate observations
    create_table_query = """
    CREATE TABLE IF NOT EXISTS exchange_rates (
    id SERIAL PRIMARY KEY,
    date DATE NOT NULL,
    currency VARCHAR(3) NOT NULL,
    rate NUMERIC,
    UNIQUE (date, currency)
    );
    """

    # Execute the table creation statement
    cursor.execute(create_table_query)

    # Commit the transaction so the table definition is persisted
    conn.commit()

#### Extract Data from the ECB API

This function sends an HTTP request to the ECB Data API using the dataset identifier provided as input.

If a `start_date` is specified, the request is restricted to observations from that date onward, enabling incremental extraction. The function returns the raw HTTP response so it can be processed in later stages of the ETL pipeline.

In [ ]:
def api_extract_data(code_data, start_date=None):

    # Define the base endpoint for the ECB Data API
    core_url = "https://data-api.ecb.europa.eu/service/data/"
    url = core_url + code_data

    # Restrict the request to observations from the specified start date
    if start_date is not None:
        url += f"?startPeriod={start_date}"

     # Headers to store the content negotiation setting for the format of the output file.
    headers = {
        "Accept": "application/vnd.sdmx.structurespecificdata+xml;version=2.1"
    }

    # Send request
    response = requests.get(url, headers=headers)
    
    # Stop execution if the HTTP response contains an error
    response.raise_for_status()

    return response

#### Transform the API response into a CSV-formatted buffer

This function parses the XML response returned by the ECB API and extracts the exchange rate observations for each available currency series.

For every observation, it collects the reference period, the currency code, and the observed value, and writes the result into an in-memory CSV buffer. The output is then returned for subsequent loading into the database.

In [ ]:
def create_csv_exchange_rate(response):

    # Parse the XML content returned by the API
    root = ET.fromstring(response.content)
    series = root.findall(".//{*}Series")

    # Store the extracted rows in an in-memory CSV buffer
    buffer = StringIO()

    for serie in series:
        currency = serie.attrib.get("CURRENCY")
        values = serie.findall(".//{*}Obs")
        for value in values:
            time_period = value.attrib.get("TIME_PERIOD")
            data_value = value.attrib.get("OBS_VALUE")
            buffer.write(f"{time_period},{currency},{data_value}\n")

    # Move the pointer to the beginning of the buffer
    buffer.seek(0)

    return buffer

#### Load exchange rate data into the database

This function reads the in-memory buffer containing exchange rate records and inserts them into the `exchange_rates` table.

Each row is expected to follow a comma-separated structure with three fields: date, currency code, and exchange rate value.

If a record with the same `date` and `currency` already exists, the function updates the existing `rate` value instead of inserting a duplicate row. This ensures that the table remains consistent and can be refreshed safely.

In [ ]:
def load_exchange_rates(buffer, conn, cursor):
    
    # Ensure the buffer is read from the beginning
    buffer.seek(0)

    # Insert new records or update existing ones when the unique key already exists
    upsert_query = """
        INSERT INTO exchange_rates (date, currency, rate)
        VALUES (%s, %s, %s)
        ON CONFLICT (date, currency)
        DO UPDATE SET rate = EXCLUDED.rate
    """
    
    # Process the buffer line by line
    for line in buffer:
        line = line.strip()

        # Skip empty lines
        if not line:
            continue

        # Split each row into its corresponding fields
        date_value, currency, rate = line.split(",")

        # Insert or update the current observation
        cursor.execute(upsert_query, (date_value, currency, rate))

    # Commit the transaction so the changes are persisted
    conn.commit()

#### Create and populate the currency dimension table

This function creates the `dim_currency` table if it does not already exist and populates it using the distinct currency codes available in the `exchange_rates` fact table.

A predefined mapping is used to associate each currency code with its corresponding currency name and country or economic area.

If a currency code already exists in the dimension table, its descriptive attributes are updated. This ensures that the dimension remains consistent and can be refreshed safely after loading new exchange rate data.

If a currency code is found in the fact table but is not defined in the mapping dictionary, the function prints a warning message.

In [ ]:
def dimension_table_currency(conn, cursor):
    
    # Map each currency code to its descriptive attributes
    currency_map = {
        "EUR": ("Euro", "Eurozone"),
        "USD": ("US Dollar", "United States"),
        "GBP": ("Pound Sterling", "United Kingdom"),
        "JPY": ("Japanese Yen", "Japan"),
        "CHF": ("Swiss Franc", "Switzerland"),
        "AUD": ("Australian Dollar", "Australia"),
        "CAD": ("Canadian Dollar", "Canada"),
        "NZD": ("New Zealand Dollar", "New Zealand"),
        "SEK": ("Swedish Krona", "Sweden"),
        "NOK": ("Norwegian Krone", "Norway"),
        "DKK": ("Danish Krone", "Denmark"),
        "PLN": ("Polish Zloty", "Poland"),
        "CZK": ("Czech Koruna", "Czech Republic"),
        "HUF": ("Hungarian Forint", "Hungary"),
        "RON": ("Romanian Leu", "Romania"),
        "BGN": ("Bulgarian Lev", "Bulgaria"),
        "ISK": ("Icelandic Krona", "Iceland")
        }


    # Create the currency dimension table if it does not already exist
    create_dim_currency =    """
        CREATE TABLE IF NOT EXISTS dim_currency (
        id VARCHAR(3) PRIMARY KEY,
        currency_name VARCHAR(50),
        country VARCHAR(50)
        ); """


    # Insert distinct currency codes extracted from the fact table
    insert_distinct_currencies = """
        INSERT INTO dim_currency (id)
        SELECT DISTINCT currency
        FROM exchange_rates
        WHERE currency IS NOT NULL
        ON CONFLICT (id) DO NOTHING;
    """


    # Retrieve all currency codes currently stored in the dimension table
    select_codes = """
        SELECT id
        FROM dim_currency;
    """

    # Insert new descriptive records or update existing ones
    upsert_currency = """
        INSERT INTO dim_currency (id, currency_name, country)
        VALUES (%s, %s, %s)
        ON CONFLICT (id)
        DO UPDATE SET
            currency_name = EXCLUDED.currency_name,
            country = EXCLUDED.country;
    """

    try:
        cursor.execute(create_dim_currency)
        cursor.execute(insert_distinct_currencies)

        cursor.execute(select_codes)
        codes = [row[0] for row in cursor.fetchall()]

        # Populate descriptive attributes for each currency code
        for code in codes:
            if code in currency_map:
                currency_name, country = currency_map[code]
                cursor.execute(upsert_currency, (code, currency_name, country))
            else:
                print(f"Currency code not mapped: {code}")

        # Commit the transaction so the dimension table is persisted
        conn.commit()

    except Exception as e:
        # Roll back the transaction if an error occurs
        conn.rollback()
        print(f"Error: {e}")
